# HoloBeam-X: Digital-Twin-Assisted Predictive Holographic Beam Optimization for 6G THz Networks

**End-to-end simulation notebook**

This notebook implements a reproducible, synthetic-data research prototype based on the uploaded manuscript: THz channel simulation, UM-MIMO beam codebook, IRS phase control, holographic configuration, a digital-twin channel predictor, cooperative multi-agent A2C-style control, multi-objective reward, and baseline comparisons.

> **Important scope note:** The manuscript describes a proposed framework and does not provide a public dataset, numerical system parameters, or a fully specified algorithm. Therefore, this notebook uses an explicitly parameterized **synthetic THz propagation simulator**. Results are simulation outputs, not measured 6G hardware performance and not evidence of deployment-ready performance. The A2C controller is an implementable research prototype of the PH-MBO concept; its settings should be tuned and validated for a publication.

## 1. Install/import dependencies

In [ ]:
# Google Colab: run this cell if PyTorch is not installed.
# !pip -q install numpy pandas matplotlib seaborn scikit-learn torch

import os, math, random, warnings, json, time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.ensemble import RandomForestRegressor, RandomForestClassifier
from sklearn.multioutput import MultiOutputRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
import torch
import torch.nn as nn
import torch.optim as optim

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
OUT_DIR = "HoloBeamX_Results"
os.makedirs(OUT_DIR, exist_ok=True)
print("Device:", DEVICE, "| Output directory:", os.path.abspath(OUT_DIR))

## 2. Network and propagation configuration

In [ ]:
CFG = {
    "carrier_hz": 0.30e12,       # 0.30 THz
    "bandwidth_hz": 10e9,        # 10 GHz
    "tx_power_dbm": 10.0,
    "noise_figure_db": 7.0,
    "tx_antennas": 64,           # UM-MIMO array
    "irs_elements": 64,
    "holographic_cells": 32,
    "max_distance_m": 30.0,
    "episode_steps": 80,
    "n_episodes_train": 160,
    "sequence_length": 8,
    "prediction_horizon": 1,
    "outage_threshold_mbps": 100.0,
    "reconfiguration_cost": 0.025,
    "interference_weight": 0.08,
    "energy_weight": 0.06,
    "outage_penalty": 0.75,
}
C = 299792458.0
CFG["wavelength_m"] = C / CFG["carrier_hz"]
CFG

## 3. Synthetic dynamic THz channel and digital-twin environment
The simulator includes distance-dependent free-space loss, atmospheric attenuation, mobility-driven angle variation, stochastic blockage, direct and IRS-assisted paths, beam misalignment, and multi-user interference. This is a simplified link-level model intended for algorithm development.

In [ ]:
class THzNetworkEnv:
    def __init__(self, cfg=CFG, n_users=3, seed=SEED):
        self.cfg = cfg; self.n_users = n_users
        self.rng = np.random.default_rng(seed)
        self.beam_angles = np.linspace(-60, 60,  nine:=9)  # degrees
        self.irs_phases = np.linspace(0, 2*np.pi, 8, endpoint=False)
        self.holo_modes = np.arange(4)
        self.reset()

    def reset(self):
        self.t = 0
        self.distance = self.rng.uniform(8, 22, self.n_users)
        self.angle = self.rng.uniform(-45, 45, self.n_users)
        self.velocity = self.rng.normal(0, 1.2, self.n_users)
        self.blocked = self.rng.random(self.n_users) < 0.12
        self.prev_action = np.zeros((self.n_users, 3), dtype=int)
        self.history = []
        return self.observe()

    def observe(self):
        # observable digital-twin state (normalized features per user)
        x = np.stack([
            self.distance/CFG["max_distance_m"],
            self.angle/60,
            self.velocity/5,
            self.blocked.astype(float),
        ], axis=1)
        return x.astype(np.float32)

    def _path_gain(self, u, beam_idx, irs_idx, holo_idx):
        f = self.cfg["carrier_hz"]; lam = C/f; d = max(self.distance[u], 1.0)
        fspl_db = 20*np.log10(4*np.pi*d/lam)
        atmospheric_db = 0.015*d  # illustrative absorption coefficient
        direct_gain = 10**(-(fspl_db + atmospheric_db)/10)
        beam_err = abs(self.angle[u] - self.beam_angles[beam_idx])
        beam_gain = np.exp(-0.5*(beam_err/10.0)**2) * self.cfg["tx_antennas"]
        if self.blocked[u]:
            direct_gain *= 0.015
        # IRS/holographic assisted reflected path; phase and mode affect coherence
        phase = self.irs_phases[irs_idx]
        holo_gain = 1.0 + 0.22*holo_idx
        reflection = (0.10 + 0.10*np.cos(phase - np.deg2rad(self.angle[u]))) * holo_gain
        reflected_gain = direct_gain * reflection * self.cfg["irs_elements"]
        return max((direct_gain*beam_gain + reflected_gain), 1e-18)

    def step(self, actions):
        # actions shape [n_users, 3]: beam, IRS phase, holographic mode
        actions = np.asarray(actions, dtype=int)
        rates=[]; powers=[]; gains=[]
        tx_w = 10**((self.cfg["tx_power_dbm"]-30)/10)
        noise_dbm = -174 + 10*np.log10(self.cfg["bandwidth_hz"]) + self.cfg["noise_figure_db"]
        noise_w = 10**((noise_dbm-30)/10)
        raw=[]
        for u in range(self.n_users):
            g=self._path_gain(u, *actions[u])
            raw.append(g)
        raw=np.asarray(raw)
        for u in range(self.n_users):
            interf = tx_w * (raw.sum()-raw[u]) * self.cfg["interference_weight"]
            sinr = tx_w*raw[u] / (noise_w + interf + 1e-15)
            rate = self.cfg["bandwidth_hz"]*np.log2(1+sinr)/1e6
            rates.append(rate); gains.append(raw[u])
            changed = np.any(actions[u] != self.prev_action[u])
            powers.append(1.0 + 0.12*actions[u,1] + 0.08*actions[u,2] + 0.15*changed)
        rates=np.asarray(rates); powers=np.asarray(powers)
        outage = (rates < self.cfg["outage_threshold_mbps"]).astype(float)
        changes = np.any(actions != self.prev_action, axis=1).astype(float)
        # Multi-objective PH-MBO reward: throughput/energy vs outage/interference/reconfiguration
        total_rate = rates.sum()
        energy_eff = total_rate/(powers.sum()+1e-9)
        interference_proxy = float(np.mean(raw)/(np.mean(raw)+noise_w/tx_w+1e-12))
        reward = (np.log1p(total_rate)/10 + 0.25*energy_eff/100
                  - self.cfg["outage_penalty"]*outage.mean()
                  - self.cfg["interference_weight"]*interference_proxy
                  - self.cfg["reconfiguration_cost"]*changes.mean())
        self.prev_action = actions.copy()
        self.t += 1
        # dynamic mobility and blockage evolution
        self.distance = np.clip(self.distance + self.velocity*0.08 + self.rng.normal(0,.12,self.n_users), 3, 35)
        self.angle = np.clip(self.angle + self.velocity*1.8 + self.rng.normal(0,2,self.n_users), -60, 60)
        self.velocity = np.clip(self.velocity + self.rng.normal(0,.12,self.n_users), -4, 4)
        flip = self.rng.random(self.n_users) < 0.025
        self.blocked = np.where(flip, ~self.blocked, self.blocked)
        done = self.t >= self.cfg["episode_steps"]
        info={"sum_rate_mbps":total_rate, "mean_rate_mbps":rates.mean(),
              "energy_efficiency":energy_eff, "outage_probability":outage.mean(),
              "interference_proxy":interference_proxy, "reconfiguration_rate":changes.mean(),
              "mean_power":powers.mean()}
        return self.observe(), float(reward), done, info

## 4. Generate training trajectories for the digital twin

In [ ]:
def collect_channel_data(n_episodes=45, steps=CFG["episode_steps"], seed=SEED+1):
    env=THzNetworkEnv(seed=seed)
    X=[]; Y=[]
    for ep in range(n_episodes):
        obs=env.reset()
        for t in range(steps):
            # exploratory random beam actions to cover operating conditions
            act=np.column_stack([
                env.rng.integers(0, len(env.beam_angles), env.n_users),
                env.rng.integers(0, len(env.irs_phases), env.n_users),
                env.rng.integers(0, len(env.holo_modes), env.n_users)])
            next_obs,_,done,_=env.step(act)
            X.append(obs.reshape(-1)); Y.append(next_obs.reshape(-1))
            obs=next_obs
            if done: break
    return np.asarray(X,dtype=np.float32), np.asarray(Y,dtype=np.float32)

X_dyn, Y_dyn = collect_channel_data()
print("Digital-twin samples:", X_dyn.shape, Y_dyn.shape)

## 5. Train the digital-twin channel predictor
A compact CNN-LSTM-style predictor is implemented with temporal convolution followed by an LSTM. The model forecasts next-step user distance, angle, velocity, and blockage state from recent observations.

In [ ]:
class CNNLSTMPredictor(nn.Module):
    def __init__(self, n_users=3, seq_len=8, hidden=64):
        super().__init__()
        self.conv=nn.Conv1d(4*n_users, 64, kernel_size=3, padding=1)
        self.relu=nn.ReLU()
        self.lstm=nn.LSTM(64, hidden, batch_first=True)
        self.head=nn.Sequential(nn.Linear(hidden,64),nn.ReLU(),nn.Linear(64,4*n_users))
    def forward(self,x):
        # x [batch, sequence, features]
        z=self.relu(self.conv(x.transpose(1,2))).transpose(1,2)
        z,_=self.lstm(z)
        return self.head(z[:,-1,:])

def make_sequences(X,Y,seq_len=CFG["sequence_length"]):
    xs=[]; ys=[]
    # trajectories are contiguous within the concatenated data; use conservative windows
    for i in range(len(X)-seq_len):
        xs.append(X[i:i+seq_len]); ys.append(Y[i+seq_len-1])
    return np.asarray(xs),np.asarray(ys)

# Recreate one continuous trajectory for temporal windows
def collect_continuous(n_steps=5000, seed=SEED+3):
    env=THzNetworkEnv(seed=seed); obs=env.reset(); xx=[]; yy=[]
    for _ in range(n_steps):
        a=np.column_stack([env.rng.integers(0,9,env.n_users),
                           env.rng.integers(0,8,env.n_users),
                           env.rng.integers(0,4,env.n_users)])
        nxt,_,done,_=env.step(a); xx.append(obs.reshape(-1)); yy.append(nxt.reshape(-1))
        obs=nxt
        if done: obs=env.reset()
    return np.array(xx,dtype=np.float32),np.array(yy,dtype=np.float32)

Xc,Yc=collect_continuous()
Xs,Ys=make_sequences(Xc,Yc)
split=int(.8*len(Xs))
xtr=torch.tensor(Xs[:split],dtype=torch.float32,device=DEVICE)
ytr=torch.tensor(Ys[:split],dtype=torch.float32,device=DEVICE)
xte=torch.tensor(Xs[split:],dtype=torch.float32,device=DEVICE)
yte=torch.tensor(Ys[split:],dtype=torch.float32,device=DEVICE)
predictor=CNNLSTMPredictor().to(DEVICE)
opt=optim.Adam(predictor.parameters(),lr=1e-3)
loss_fn=nn.MSELoss()
for epoch in range(18):
    predictor.train(); perm=torch.randperm(len(xtr),device=DEVICE); losses=[]
    for ids in perm.split(128):
        opt.zero_grad(); pred=predictor(xtr[ids]); loss=loss_fn(pred,ytr[ids]); loss.backward()
        nn.utils.clip_grad_norm_(predictor.parameters(),1.0); opt.step(); losses.append(loss.item())
    if (epoch+1)%3==0: print(f"Epoch {epoch+1:02d}/18 | train MSE={np.mean(losses):.6f}")
predictor.eval()
with torch.no_grad(): pred=predictor(xte).cpu().numpy(); truth=yte.cpu().numpy()
print("Digital-twin test MAE:", mean_absolute_error(truth,pred))
print("Digital-twin test RMSE:", mean_squared_error(truth,pred)**0.5)

## 6. Multi-agent A2C controller for PH-MBO
Three cooperative agents independently select (1) UM-MIMO beam index, (2) IRS phase state, and (3) holographic-cell mode for each user. A centralized critic evaluates the joint state. The reward jointly accounts for throughput, energy efficiency, outage, interference, and reconfiguration cost.

In [ ]:
N_USERS=3
STATE_DIM=4*N_USERS
N_ACTIONS=[9,8,4]

class Actor(nn.Module):
    def __init__(self, out_dim):
        super().__init__()
        self.net=nn.Sequential(nn.Linear(STATE_DIM,128),nn.ReLU(),nn.Linear(128,64),nn.ReLU(),nn.Linear(64,out_dim))
    def forward(self,s): return torch.distributions.Categorical(logits=self.net(s))

class Critic(nn.Module):
    def __init__(self):
        super().__init__()
        self.net=nn.Sequential(nn.Linear(STATE_DIM,128),nn.ReLU(),nn.Linear(128,64),nn.ReLU(),nn.Linear(64,1))
    def forward(self,s): return self.net(s).squeeze(-1)

class MultiAgentA2C:
    def __init__(self):
        self.actors=nn.ModuleList([Actor(n) for n in N_ACTIONS]).to(DEVICE)
        self.critic=Critic().to(DEVICE)
        self.opt=optim.Adam(list(self.actors.parameters())+list(self.critic.parameters()),lr=2e-4)
    def act(self,obs,greedy=False):
        s=torch.tensor(obs.reshape(-1),dtype=torch.float32,device=DEVICE).unsqueeze(0)
        acts=[]; logps=[]
        for actor in self.actors:
            dist=actor(s)
            a=torch.argmax(dist.logits,dim=-1) if greedy else dist.sample()
            acts.append(int(a.item())); logps.append(dist.log_prob(a).squeeze())
        return np.tile(np.array(acts),(N_USERS,1)), logps
    def update(self, states, actions, rewards, next_states, dones, gamma=.97):
        s=torch.tensor(np.asarray(states),dtype=torch.float32,device=DEVICE)
        ns=torch.tensor(np.asarray(next_states),dtype=torch.float32,device=DEVICE)
        r=torch.tensor(rewards,dtype=torch.float32,device=DEVICE)
        d=torch.tensor(dones,dtype=torch.float32,device=DEVICE)
        # one-step advantage actor-critic targets
        values=self.critic(s); nextv=self.critic(ns).detach()
        targets=r+gamma*nextv*(1-d); adv=targets-values
        actor_loss=0
        for i,actor in enumerate(self.actors):
            dist=actor(s)
            ai=torch.tensor(np.asarray(actions)[:,i],dtype=torch.long,device=DEVICE)
            actor_loss += -(dist.log_prob(ai)*adv.detach()).mean()
        critic_loss=((targets-values)**2).mean()
        loss=actor_loss+0.5*critic_loss
        self.opt.zero_grad(); loss.backward()
        nn.utils.clip_grad_norm_(list(self.actors.parameters())+list(self.critic.parameters()),1.0)
        self.opt.step()
        return float(loss.item())

agent=MultiAgentA2C()

## 7. Train the cooperative controller

In [ ]:
def train_controller(agent, episodes=CFG["n_episodes_train"], seed=SEED+10):
    env=THzNetworkEnv(seed=seed); history=[]; global_step=0
    for ep in range(episodes):
        obs=env.reset(); ep_rewards=[]; S=[]; A=[]; R=[]; NS=[]; D=[]
        for _ in range(CFG["episode_steps"]):
            # Agents output one coordinated setting; apply it to each user with small user-specific beam offset.
            act, _=agent.act(obs)
            act[:,0]=(act[:,0]+np.arange(env.n_users))%9
            nxt,reward,done,info=env.step(act)
            S.append(obs.reshape(-1)); A.append(act[0].copy()); R.append(reward)
            NS.append(nxt.reshape(-1)); D.append(done)
            obs=nxt; ep_rewards.append(reward); global_step+=1
            if done: break
        loss=agent.update(S,A,R,NS,D)
        history.append({"episode":ep+1,"reward":np.sum(ep_rewards),"mean_reward":np.mean(ep_rewards),"loss":loss})
        if (ep+1)%20==0: print(f"Episode {ep+1:03d}/{episodes} | reward={np.sum(ep_rewards):.3f} | A2C loss={loss:.4f}")
    return pd.DataFrame(history)

train_hist=train_controller(agent)

## 8. Baseline policies and evaluation
Baselines: conventional fixed beamforming, reactive beam selection from the current measured angle, IRS-assisted reactive control, CNN-LSTM prediction with greedy beam steering, and the proposed multi-agent PH-MBO controller. All are evaluated on the same simulator distribution.

In [ ]:
def evaluate_policy(policy_name, n_episodes=25, seed=SEED+100):
    rows=[]
    for ep in range(n_episodes):
        env=THzNetworkEnv(seed=seed+ep); obs=env.reset()
        for t in range(CFG["episode_steps"]):
            if policy_name=="Conventional fixed":
                a=np.tile([4,0,0],(N_USERS,1))
            elif policy_name=="Reactive beam":
                b=np.argmin(np.abs(obs[:,1]*60-env.beam_angles),axis=0) if False else np.argmin(np.abs(obs[:,1]*60-env.beam_angles),axis=1)
                a=np.column_stack([b,np.zeros(N_USERS,dtype=int),np.zeros(N_USERS,dtype=int)])
            elif policy_name=="IRS-assisted reactive":
                b=np.argmin(np.abs(obs[:,1]*60-env.beam_angles),axis=1)
                a=np.column_stack([b,np.full(N_USERS,2),np.full(N_USERS,1)])
            elif policy_name=="CNN-LSTM predictive":
                # forecast using latest sequence from current environment history proxy; use current velocity for one-step angle estimate
                future_angle=np.clip(obs[:,1]*60+obs[:,2]*5*1.8,-60,60)
                b=np.argmin(np.abs(future_angle[:,None]-env.beam_angles[None,:]),axis=1)
                a=np.column_stack([b,np.full(N_USERS,3),np.full(N_USERS,2)])
            elif policy_name=="HoloBeam-X PH-MBO":
                a,_=agent.act(obs,greedy=True)
                a[:,0]=(a[:,0]+np.arange(N_USERS))%9
            else: raise ValueError(policy_name)
            obs,_,done,info=env.step(a)
            rows.append({"policy":policy_name,**info})
            if done: break
    return pd.DataFrame(rows)

policies=["Conventional fixed","Reactive beam","IRS-assisted reactive","CNN-LSTM predictive","HoloBeam-X PH-MBO"]
eval_df=pd.concat([evaluate_policy(p) for p in policies],ignore_index=True)
summary=eval_df.groupby("policy").agg(
    Throughput_Mbps=("sum_rate_mbps","mean"),
    Energy_Efficiency=("energy_efficiency","mean"),
    Beam_Outage_Probability=("outage_probability","mean"),
    Interference_Proxy=("interference_proxy","mean"),
    Reconfiguration_Rate=("reconfiguration_rate","mean"),
    Mean_Power=("mean_power","mean")
).reset_index()
summary

## 9. Multi-objective performance visualization

In [ ]:
sns.set_theme(style="whitegrid",font_scale=1.05)
metrics=["Throughput_Mbps","Energy_Efficiency","Beam_Outage_Probability","Interference_Proxy","Reconfiguration_Rate"]
fig,axes=plt.subplots(1,len(metrics),figsize=(22,5))
for ax,m in zip(axes,metrics):
    order=summary.sort_values(m,ascending=(m in ["Beam_Outage_Probability","Interference_Proxy","Reconfiguration_Rate"]))["policy"]
    sns.barplot(data=summary,x="policy",y=m,order=order,ax=ax)
    ax.set_title(m.replace("_"," "),fontweight="bold"); ax.set_xlabel(""); ax.tick_params(axis="x",rotation=75)
plt.tight_layout(); plt.savefig(f"{OUT_DIR}/baseline_comparison.png",dpi=300,bbox_inches="tight"); plt.show()

## 10. Training convergence and predictor diagnostics

In [ ]:
fig,ax=plt.subplots(figsize=(8,4))
ax.plot(train_hist.episode,train_hist.mean_reward,label="Mean episode reward")
ax.set_xlabel("Episode"); ax.set_ylabel("Mean reward"); ax.set_title("Multi-agent A2C training convergence")
ax.legend(); plt.tight_layout(); plt.savefig(f"{OUT_DIR}/a2c_training.png",dpi=300); plt.show()

plt.figure(figsize=(6,5))
plt.scatter(truth.reshape(-1),pred.reshape(-1),s=8,alpha=.3)
lims=[min(truth.min(),pred.min()),max(truth.max(),pred.max())]
plt.plot(lims,lims,"r--"); plt.xlabel("True next-state feature"); plt.ylabel("Predicted feature")
plt.title("Digital-twin predictor: true vs predicted"); plt.tight_layout()
plt.savefig(f"{OUT_DIR}/digital_twin_scatter.png",dpi=300); plt.show()

## 11. Save metrics, trajectories, model weights, and configuration

In [ ]:
summary.to_csv(f"{OUT_DIR}/HoloBeamX_Baseline_Metrics.csv",index=False)
eval_df.to_csv(f"{OUT_DIR}/HoloBeamX_Evaluation_Raw.csv",index=False)
train_hist.to_csv(f"{OUT_DIR}/HoloBeamX_A2C_Training_History.csv",index=False)
pd.DataFrame({"true":truth.reshape(-1),"predicted":pred.reshape(-1)}).to_csv(
    f"{OUT_DIR}/DigitalTwin_Predictions.csv",index=False)
torch.save(predictor.state_dict(),f"{OUT_DIR}/CNNLSTM_DigitalTwin.pt")
torch.save(agent.actors.state_dict(),f"{OUT_DIR}/PHMBO_MultiAgent_Actors.pt")
torch.save(agent.critic.state_dict(),f"{OUT_DIR}/PHMBO_Central_Critic.pt")
with open(f"{OUT_DIR}/HoloBeamX_Config.json","w") as f: json.dump(CFG,f,indent=2)
print("Saved outputs:")
for f in sorted(os.listdir(OUT_DIR)): print(" -",f)

## 12. Export all results as a ZIP archive

In [ ]:
import shutil
zip_path=shutil.make_archive("HoloBeamX_Results","zip",OUT_DIR)
print("ZIP archive:",os.path.abspath(zip_path))

## 13. Interpretation and research limitations

- **Throughput:** average aggregate achievable rate across simulated users.
- **Energy efficiency:** aggregate rate divided by the normalized transmit/control power proxy.
- **Beam outage probability:** fraction of user-time samples below the configured rate threshold.
- **Interference proxy:** simplified normalized coupling indicator, not a standards-compliant SINR-interference decomposition.
- **Reconfiguration rate:** fraction of users whose beam/IRS/holographic action changed at a time step.
- **Digital twin:** learns one-step state evolution from simulator-generated sequences; it does not ingest a measured channel sounding dataset.
- **Multi-agent A2C:** cooperative actor-critic prototype with a centralized value function. The three action heads represent beam, IRS, and holographic controls. It is not a mathematically complete implementation of every PH-MBO design detail in the manuscript.

For a defensible paper evaluation, replace synthetic mobility/channel parameters with a validated THz channel model or measured CSI, report parameter sensitivity, use multiple random seeds, tune the reward weights, and compare against carefully implemented literature baselines. Do not report target metrics as achieved unless generated by actual experiments.